In [ ]:
import random
import numpy as np
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.vec_env import VecNormalize

from agent import FloodWarningEnv 

# Seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# Training environment
env = make_vec_env(FloodWarningEnv, n_envs=8, seed=SEED)
#env = VecNormalize(env, norm_obs=True, norm_reward=False)

# Evaluation environment
eval_env = FloodWarningEnv()
eval_env.reset(seed=SEED)

# Agent setup
model = PPO(
    "MlpPolicy", # MLP as the policy, takes observation vector as input and outputs probability distribution over the 4 warning levels
    env,
    learning_rate=3e-4, # how fast network weights update
    n_steps=2048, # how many step collected across all environments before a policy update (total experience per update = 2028 * 8 = 16384 samples)
    batch_size=64, # how many samples used in each gradient update within a single policy update cycle
    n_epochs=10, # how many times collected experience is reused for gradient updates before being discarded 
    gamma=0.99, # discount factor for future rewards
    verbose=1, # prints training progress
    tensorboard_log="./logs/"
)

# Evaluation and model saving during training
eval_callback = EvalCallback( # periodically pauses training, runs agent deterministically on seperate evaluation environment, saves best performing model
    eval_env,
    best_model_save_path="./models/best_model/",
    eval_freq=10_000, # evaluations runs every 10000 steps
    deterministic=True,
)

# Executes training, with evaluation running
model.learn(total_timesteps=1_000_000, callback=eval_callback) # runs training loop for 1 million environment steps, calling callback at the specified frequency

# Save final model (saved final weights at the end of training for resuming training)
model.save("./models/final_model")